# 🤖 Object Detection + Distance Estimation — Exploration Notebook

This notebook walks through:
1. Dataset EDA (class distribution, sample images)
2. Distance estimation geometry
3. Training results (loss curves, mAP)
4. Qualitative results (annotated detections)
5. Optimization comparison (FPS before/after)
6. Extra credit: Bird's-eye view + Optical flow

In [ ]:
import sys
sys.path.insert(0, '..')

import cv2
import numpy as np
import matplotlib.pyplot as plt
import yaml
from pathlib import Path

with open('../configs/config.yaml') as f:
    cfg = yaml.safe_load(f)

print('Config loaded ✓')

## 1. Dataset EDA

In [ ]:
from src.dataset import BDD100KNavDataset

ds = BDD100KNavDataset(cfg)

# Class distribution after conversion
for split in ('train', 'val'):
    dist = ds.class_distribution(split)
    print(f'\n{split.upper()} class distribution:')
    for cls, cnt in dist.items():
        bar = '█' * (cnt // 200)
        print(f'  {cls:<16} {cnt:>6}  {bar}')

In [ ]:
# Plot class distribution
dist_train = ds.class_distribution('train')

fig, ax = plt.subplots(figsize=(7, 4))
colors = ['#FF8C00', '#FFD700', '#DC143C']
ax.bar(dist_train.keys(), dist_train.values(), color=colors, edgecolor='black')
ax.set_title('BDD100K — Navigation Class Distribution (Train)', fontsize=13)
ax.set_ylabel('Instance count')
ax.set_xlabel('Class')
plt.tight_layout()
plt.savefig('../results/class_distribution.png', dpi=150)
plt.show()

## 2. Distance Estimation — Geometry Intuition

In [ ]:
from src.distance import MonoDistanceEstimator
import numpy as np

est = MonoDistanceEstimator(cfg)
print(f'Focal length: {est.focal_px} px')
print(f'Image size  : {est.img_w} × {est.img_h}')

# Simulate bbox heights at various true distances
distances_m   = np.linspace(1, 40, 200)
real_height_m = 0.75  # traffic cone
bbox_heights  = (real_height_m * est.focal_px) / distances_m  # px

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(distances_m, bbox_heights, color='#FF8C00', lw=2)
axes[0].set_xlabel('True distance (m)')
axes[0].set_ylabel('Bounding box height (px)')
axes[0].set_title('BBox height vs. distance (cone, f=800px)')
axes[0].grid(True, alpha=0.3)

# Estimated vs true distance (with ±10% focal-length error)
est_dist_nominal = (real_height_m * est.focal_px) / bbox_heights
est_dist_p10     = (real_height_m * est.focal_px * 1.1) / bbox_heights
est_dist_m10     = (real_height_m * est.focal_px * 0.9) / bbox_heights

axes[1].plot(distances_m, est_dist_nominal, 'b-',  lw=2, label='nominal f')
axes[1].fill_between(distances_m, est_dist_m10, est_dist_p10,
                     alpha=0.2, color='blue', label='±10% f error')
axes[1].plot(distances_m, distances_m, 'k--', lw=1, label='ground truth')
axes[1].set_xlabel('True distance (m)')
axes[1].set_ylabel('Estimated distance (m)')
axes[1].set_title('Estimation error sensitivity to focal length')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../results/distance_geometry.png', dpi=150)
plt.show()

## 3. Sample Annotated Detections

In [ ]:
from src.inference import InferencePipeline

WEIGHTS = '../runs/train/weights/best.pt'  # update if needed

pipeline = InferencePipeline(cfg, weights=WEIGHTS)

# Pick a few val images
val_imgs = list(Path(cfg['dataset']['images_val']).glob('*.jpg'))[:4]

fig, axes = plt.subplots(2, 2, figsize=(16, 9))
for ax, img_path in zip(axes.flat, val_imgs):
    annotated = pipeline.run_image(str(img_path), out_path=None)
    ax.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    ax.set_title(img_path.name, fontsize=8)
    ax.axis('off')
plt.suptitle('Sample Annotated Detections (with distance)', fontsize=13)
plt.tight_layout()
plt.savefig('../results/sample_detections.png', dpi=150)
plt.show()

## 4. Optimization: Before vs After

In [ ]:
from src.optimize import ModelOptimizer
import pandas as pd

opt = ModelOptimizer(cfg, weights_path=WEIGHTS)

results = []
for variant, path, dev in [
    ('YOLOv8n FP32 (PT)',  WEIGHTS,                                      'cpu'),
    ('ONNX FP32',          '../runs/model_fp32.onnx',                    'cpu'),
    ('ONNX INT8 (quant)',   '../runs/model_int8.onnx',                    'cpu'),
]:
    if not Path(path).exists():
        continue
    m = opt.benchmark_fps(path, n_runs=50, device=dev)
    results.append({'Model': variant, 'Device': dev,
                    'FPS': round(m['fps'], 1), 'Avg ms': round(m['avg_ms'], 1)})

df = pd.DataFrame(results)
print(df.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(df['Model'], df['FPS'], color=['#3A86FF','#8338EC','#FF006E'])
ax.bar_label(bars, fmt='%.1f fps', padding=5)
ax.set_xlabel('Frames per Second (higher is better)')
ax.set_title('Before vs. After Optimization — CPU FPS')
ax.set_xlim(0, df['FPS'].max() * 1.25)
plt.tight_layout()
plt.savefig('../results/optimization_fps.png', dpi=150)
plt.show()

## 5. Extra Credit — Bird's-Eye View

In [ ]:
from src.annotator import Annotator

ann = Annotator(cfg)
sample_path = str(val_imgs[0]) if val_imgs else '../data/sample.jpg'
frame = cv2.imread(sample_path)

if frame is not None:
    bev = ann.bird_eye_view(frame)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    ax1.set_title('Original (front-camera view)')
    ax1.axis('off')
    ax2.imshow(cv2.cvtColor(bev, cv2.COLOR_BGR2RGB))
    ax2.set_title('Bird's-Eye View (homography warp)')
    ax2.axis('off')
    plt.tight_layout()
    plt.savefig('../results/bird_eye_view.png', dpi=150)
    plt.show()